In [ ]:
import scvelo as scv
import scanpy as sc
from multiprocessing import Pool
import anndata as ad
import numpy as np
import pandas as pd
from matplotlib import rcParams

In [ ]:
import mudata as mu

In [ ]:
import scanpy as sc

scv.settings.figdir = "/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/prj_honda_15062026/publication_figures/healthy"


In [ ]:
adata = ad.read_h5ad("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/single_cell_normal/002_annotate_adata_normal.h5ad")


In [ ]:
adata = adata[~adata.obs_names.duplicated(), :]

In [ ]:
adata

In [ ]:
import os
import glob
import scvelo as scv
import scanpy as sc

def read_scvelo_2019_looms():
    loom_dir = "/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/trajectory_inference/loom"
    
    # find all loom files starting with 2021
    loom_files = sorted(
        glob.glob(os.path.join(loom_dir, "2019*.loom"))
    )
    
    print(f"Found {len(loom_files)} loom files.")
    
    adatas = []
    
    for file in loom_files:
        print(f"Reading {os.path.basename(file)}")
        
        adata = scv.read_loom(file)
        adata.var_names_make_unique()
        
        # store filename (without extension) as metadata
        sample_name = os.path.basename(file).replace(".loom", "")
        adata.obs["sample_id"] = sample_name
        
        adatas.append(adata)
    
    # concatenate all
    adata_combined = sc.concat(adatas, join="outer", label="batch", keys=[a.obs["sample_id"][0] for a in adatas])
    
    return adata_combined


In [ ]:
adata_velocity = read_scvelo_2019_looms()

In [ ]:
adata_scvelo = adata_velocity

In [ ]:
adata_scvelo

In [ ]:
adata.obs_names

In [ ]:
import re

# Step 1: extract clean barcode
barcodes = (
    adata_scvelo.obs_names
    .str.split(":").str[1]      # take part after :
    .str.replace("x", "", regex=False)  # remove trailing x
)

# Step 2: clean sample names
samples = (
    adata_scvelo.obs["sample_id"]
    .str.replace("2019_", "", regex=False)
)

# Step 3: build matching obs_names
new_names = samples + "_" + barcodes + "-1"

adata_scvelo.obs_names = new_names


In [ ]:
adata_scvelo.obs

In [ ]:
adata.obs_names

In [ ]:
len(set(adata.obs_names)), len(set(adata_scvelo.obs_names)), len(
    set(adata.obs_names) & set(adata_scvelo.obs_names)
)

In [ ]:
adata_scvelo = scv.utils.merge(adata_scvelo, adata)

In [ ]:
scv.pp.filter_and_normalize(adata_scvelo)

In [ ]:
scv.pp.moments(adata_scvelo, n_pcs=30, n_neighbors=30)

In [ ]:
scv.tl.velocity(adata_scvelo)

In [ ]:
scv.tl.velocity_graph(adata_scvelo)

In [ ]:
adata_scvelo

In [ ]:
scv.pl.proportions(adata_scvelo, groupby="sample_id")

In [ ]:
sc.set_figure_params(figsize=(10, 10))
rcParams["axes.grid"] = False


In [ ]:
adata

In [ ]:
ax = sc.pl.embedding(
    adata,
    basis="X_umap",
    color="rna_leiden_12",
    show=False,
    legend_loc="None",
    size=45,
    alpha=0.3,frameon=False
)

scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="X_umap",
    color="rna_leiden_12",arrow_size=2,linewidth=2,
    #     arrow_color="white",
    legend_loc="right margin",
    ax=ax,
    alpha=0, save = "velocity_normal_rna_leiden_12.png"
    #save = "velocity_normal_rna_leiden_12.svg"
)

ax.get_figure()

In [ ]:
ax = sc.pl.embedding(
    adata,
    basis="X_umap",
    color="cell_annotation_05",
    show=False,
    legend_loc="None",
    size=45,
    alpha=0.3,frameon=False
)

scv.pl.velocity_embedding_stream(
    adata_scvelo,
    basis="X_umap",
    color="cell_annotation_05",arrow_size=2,linewidth=2,
    #     arrow_color="white",
    legend_loc="right margin",
    ax=ax,
    alpha=0, save ="velocity_normal.png" #save = "velocity_normal.svg"
)

ax.get_figure()